In [1]:
# Google Colab-only setup — run this notebook in its own fresh Colab runtime.
import sys
if "google.colab" not in sys.modules:
    raise RuntimeError(
        "This session 1 is Google Colab-only. Open https://colab.research.google.com/, "
        "upload this notebook, and run it there."
    )

%pip install -q ultralytics==8.4.102 matplotlib numpy pillow

import torch
torch.manual_seed(0)


RuntimeError: This session 1 is Google Colab-only. Open https://colab.research.google.com/, upload this notebook, and run it there.

In [2]:
# Independent Colab assets — this notebook never reads another session's files.
from pathlib import Path
import os

SESSION_WORKSPACE = Path("/content/yolo_object_detection_session_01")
SESSION_WORKSPACE.mkdir(parents=True, exist_ok=True)
os.chdir(SESSION_WORKSPACE)

from ultralytics import YOLO, settings
from ultralytics.data.utils import check_det_dataset

SESSION_DATASETS = SESSION_WORKSPACE / "datasets"
settings.update({"datasets_dir": str(SESSION_DATASETS)})
COCO128_INFO = check_det_dataset("coco128.yaml", autodownload=True)
COCO128_YAML = Path(COCO128_INFO.get("yaml_file", "coco128.yaml"))
BASELINE_MODEL = YOLO("yolo11n.pt")  # Downloads and caches this notebook's pretrained weights.
print(f"Colab-only session 1: workspace={SESSION_WORKSPACE} | dataset={COCO128_YAML} | model=yolo11n.pt")


PermissionError: [Errno 13] Permission denied: '/content'

## 1. Record the environment

A result is easier to reproduce when its package version, checkpoint, device, and random seed are recorded.

In [ ]:
from pathlib import Path
from importlib.metadata import PackageNotFoundError, version
import os
import random
import tempfile
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image, ImageDraw

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
IS_COLAB = "google.colab" in sys.modules
CHECKPOINT = os.environ.get("YOLO_CHECKPOINT", "yolo11n.pt")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
try:
    ULTRALYTICS_VERSION = version("ultralytics")
    ULTRALYTICS_AVAILABLE = True
except PackageNotFoundError:
    ULTRALYTICS_VERSION = "not installed (fallback mode)"
    ULTRALYTICS_AVAILABLE = False

print({
    "seed": SEED, "python": sys.version.split()[0], "torch": torch.__version__,
    "ultralytics": ULTRALYTICS_VERSION, "checkpoint": CHECKPOINT, "device": DEVICE
})

## 2. Inventory downloaded COCO128

COCO128 follows the Ultralytics detection layout: a dataset YAML names the classes and train split, `images/` holds photographs, and same-stem files under `labels/` hold one object per row. This notebook uses the downloaded COCO128 labels directly.

In [ ]:
# Use only the COCO128 copy downloaded by this notebook's Colab bootstrap.
YAML_PATH = COCO128_YAML
IMAGE_DIR = Path(COCO128_INFO["train"])
COCO_ROOT = Path(COCO128_INFO["path"])
relative_split = IMAGE_DIR.relative_to(COCO_ROOT / "images")
LABEL_DIR = COCO_ROOT / "labels" / relative_split
CLASS_NAMES = COCO128_INFO["names"]
if isinstance(CLASS_NAMES, list):
    CLASS_NAMES = dict(enumerate(CLASS_NAMES))
DATA_MODE = "COCO128 downloaded in this notebook"

print("Dataset mode:", DATA_MODE)
print("YAML:", YAML_PATH)
print("Images:", IMAGE_DIR)
print("Labels:", LABEL_DIR)
print("Classes:", CLASS_NAMES)
# Build only verified image–label pairs. COCO128 archives can contain orphan labels,
# so never derive an image path from a label without checking it exists first.
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}
image_by_stem = {path.stem: path for path in IMAGE_DIR.rglob("*") if path.suffix.lower() in IMAGE_SUFFIXES}
all_label_paths = sorted(LABEL_DIR.rglob("*.txt"))
label_paths = [path for path in all_label_paths if path.stem in image_by_stem]
missing_image_labels = [path for path in all_label_paths if path.stem not in image_by_stem]
image_paths = [image_by_stem[path.stem] for path in label_paths]
all_rows = []
for label_path in label_paths:
    for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        if line.strip():
            all_rows.append((label_path, line_number, line.strip()))

print(f"Verified {len(image_paths)} image–label pairs; skipped {len(missing_image_labels)} orphan label file(s).")
print(f"Found {len(all_rows)} label rows in verified pairs.")
print("Three raw label examples:")
for path, line_number, row in all_rows[:3]:
    print(f"  {path.name}:{line_number}  {row}")
assert len(image_paths) >= 3 and len(all_rows) >= 3, "COCO128 download is incomplete."


In [ ]:
# Build only verified image–label pairs. COCO128 archives can contain orphan labels,
# so never derive an image path from a label without checking it exists first.
IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png"}
image_by_stem = {path.stem: path for path in IMAGE_DIR.rglob("*") if path.suffix.lower() in IMAGE_SUFFIXES}
all_label_paths = sorted(LABEL_DIR.rglob("*.txt"))
label_paths = [path for path in all_label_paths if path.stem in image_by_stem]
missing_image_labels = [path for path in all_label_paths if path.stem not in image_by_stem]
image_paths = [image_by_stem[path.stem] for path in label_paths]
all_rows = []
for label_path in label_paths:
    for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        if line.strip():
            all_rows.append((label_path, line_number, line.strip()))

print(f"Verified {len(image_paths)} image–label pairs; skipped {len(missing_image_labels)} orphan label file(s).")
print(f"Found {len(all_rows)} label rows in verified pairs.")
print("Three raw label examples:")
for path, line_number, row in all_rows[:3]:
    print(f"  {path.name}:{line_number}  {row}")
assert len(image_paths) >= 3 and len(all_rows) >= 3, "COCO128 download is incomplete."


## 3. Decode one label by hand

For a `640 × 480` image, consider `2 0.5 0.5 0.5 0.5`. The normalized center becomes `(320, 240)` pixels and the size becomes `(320, 240)` pixels. Half the width/height extends on each side, so the corners are `(160, 120, 480, 360)`. No `+1` is used because we treat boxes as half-open geometric rectangles.

In [ ]:
supplied = "2 0.5 0.5 0.5 0.5"
_, x_center_n, y_center_n, width_n, height_n = supplied.split()
W, H = 640, 480
x_center, y_center = float(x_center_n) * W, float(y_center_n) * H
box_width, box_height = float(width_n) * W, float(height_n) * H
hand_corners = (x_center-box_width/2, y_center-box_height/2, x_center+box_width/2, y_center+box_height/2)
print(f"center=({x_center:.0f}, {y_center:.0f}), size=({box_width:.0f}, {box_height:.0f})")
print("corners (x1, y1, x2, y2) =", hand_corners)
assert hand_corners == (160.0, 120.0, 480.0, 360.0)

## 4. Exercise E1 — convert normalized center-size to pixel corners

Complete the four equations. Keep floating-point corners: rounding is a separate display decision.

In [ ]:
# TODO: Return (x1, y1, x2, y2) in pixels from normalized center-size values.
# HINT: Multiply x/width quantities by image_width and y/height by image_height.


## 5. Exercise E2 — validate before decoding

A decoder should fail loudly on bad annotations. Add checks for the five-field layout, class ID, finite normalized values, allowed range, and positive dimensions. A valid box is asserted to stay inside the image instead of being silently clipped.

In [ ]:
# TODO: Parse and validate one YOLO row, then return (class_id, x1, y1, x2, y2).
# HINT: Reject wrong field counts, invalid class IDs/ranges, non-finite values, and nonpositive sizes.


In [ ]:
# Audit every nonempty label row programmatically.
for label_path, line_number, row in all_rows:
    with Image.open(image_by_stem[label_path.stem]) as image:
        decode_yolo_row(row, image.width, image.height, len(CLASS_NAMES))
print(f"Validated all {len(all_rows)} rows.")

# Malformed examples should trigger clear assertions.
malformed = {
    "wrong field count": "0 0.5 0.5 0.2",
    "invalid class": f"{len(CLASS_NAMES)} 0.5 0.5 0.2 0.2",
    "range error": "0 1.2 0.5 0.2 0.2",
    "nonpositive width": "0 0.5 0.5 0.0 0.2",
}
for description, row in malformed.items():
    try:
        decode_yolo_row(row, 640, 480, len(CLASS_NAMES))
        raise RuntimeError(f"Malformed row unexpectedly passed: {description}")
    except AssertionError as error:
        print(f"Caught {description}: {error}")

## 6. Visualize three ground-truth label sets

The picture is the final geometry check. A mathematically valid box can still surround the wrong object if files were paired incorrectly.

In [ ]:
def matching_label_path(image_path):
    return LABEL_DIR / f"{image_path.stem}.txt"

def ground_truth_for(image_path):
    with Image.open(image_path) as opened:
        width, height = opened.size
    label_path = matching_label_path(image_path)
    rows = label_path.read_text(encoding="utf-8").splitlines() if label_path.exists() else []
    return [decode_yolo_row(row, width, height, len(CLASS_NAMES)) for row in rows if row.strip()]

selected_images = [path for path in image_paths if matching_label_path(path).exists()][:3]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, image_path in zip(axes, selected_images):
    image = Image.open(image_path).convert("RGB")
    axis.imshow(image); axis.set_title(image_path.name); axis.axis("off")
    for class_id, x1, y1, x2, y2 in ground_truth_for(image_path):
        axis.add_patch(plt.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, ec="#f28e2b", lw=2.5))
        axis.text(x1, y1, CLASS_NAMES[class_id], color="white", fontsize=9,
                  bbox={"facecolor": "#333333", "alpha": 0.8, "pad": 2})
plt.tight_layout()

## 7. Run pretrained inference on the same images

A detection is one `(class, confidence, xyxy)` record. The number of records may differ across images—even zero is valid. In fallback mode, deterministic pseudo-predictions keep the output contract visible; they are **not** model results.

In [ ]:
CONFIDENCE_THRESHOLD = 0.25
checkpoint_path = Path(CHECKPOINT).expanduser()
REAL_INFERENCE = ULTRALYTICS_AVAILABLE and (IS_COLAB or checkpoint_path.exists())

if REAL_INFERENCE:
    from ultralytics import YOLO
    model = YOLO(CHECKPOINT)
    raw_results = model.predict([str(path) for path in selected_images], conf=CONFIDENCE_THRESHOLD, device=DEVICE, verbose=False)
    prediction_rows = []
    rendered_images = []
    for image_path, result in zip(selected_images, raw_results):
        rows = []
        for box in result.boxes:
            class_id = int(box.cls.item())
            confidence = float(box.conf.item())
            corners = tuple(float(value) for value in box.xyxy[0].tolist())
            rows.append((result.names[class_id], confidence, corners))
        prediction_rows.append(rows)
        rendered_images.append(result.plot()[..., ::-1])  # Ultralytics BGR -> RGB
    INFERENCE_MODE = f"Ultralytics {ULTRALYTICS_VERSION}, {CHECKPOINT}"
else:
    prediction_rows = []
    rendered_images = []
    for image_path in selected_images:
        image = Image.open(image_path).convert("RGB")
        draw = ImageDraw.Draw(image)
        rows = []
        for index, (class_id, x1, y1, x2, y2) in enumerate(ground_truth_for(image_path)):
            # Fixed offsets make a visible, reproducible label/prediction disagreement.
            dx, dy = 5 + 2*index, 3 + index
            corners = (x1+dx, y1+dy, x2+dx, y2+dy)
            confidence = 0.88 - 0.09*index
            rows.append((CLASS_NAMES[class_id], confidence, corners))
            draw.rectangle(corners, outline=(170, 40, 40), width=3)
            draw.text((corners[0], max(0, corners[1]-12)), f"{CLASS_NAMES[class_id]} {confidence:.2f}", fill=(120, 0, 0))
        prediction_rows.append(rows)
        rendered_images.append(np.asarray(image))
    INFERENCE_MODE = "deterministic pseudo-predictions (offline validation only)"

print("Inference mode:", INFERENCE_MODE)
print("Confidence threshold:", CONFIDENCE_THRESHOLD)
for image_path, rows in zip(selected_images, prediction_rows):
    print(f"\n{image_path.name}: {len(rows)} detection(s)")
    for class_name, confidence, corners in rows:
        rounded = tuple(round(value, 1) for value in corners)
        print(f"  class={class_name:<12} confidence={confidence:.3f} xyxy={rounded}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, image_path, rendered in zip(axes, selected_images, rendered_images):
    axis.imshow(rendered); axis.set_title(f"{image_path.name}\n{len(prediction_rows[axes.tolist().index(axis)])} prediction(s)"); axis.axis("off")
plt.tight_layout()

## 8. Compare the three tasks

Describe their outputs and annotation types, not just their applications.

> TODO: Write exactly three sentences: one each for classification, semantic segmentation, and detection.


## 9. Exercise E3 — explain one disagreement

Compare one label and prediction. Do not declare either one wrong before checking the image, annotation policy, threshold, class mapping, and localization evidence.

> TODO: Explain one label/prediction disagreement in 2–3 evidence-aware sentences.


## Wrap-up

You audited YOLO's five-field annotation format, converted normalized center-size values to pixel corners, rejected malformed rows, visualized three ground-truth examples, and inspected variable-length detection records. The central habit is simple: **check the label language visually before interpreting model behavior**.

**Concept check:** Why is `0.5` a different geometric quantity in the second, fourth, and fifth fields of `2 0.5 0.5 0.5 0.5`?